# Financial Sentiment Analysis — From Tweet Classification to Trading Signals

**Author:** Tommaso Zeri  
**Course:** Advanced Machine Learning — Università di Bologna  
**Environment:** Google Colab Pro (A100 40GB)

This notebook implements a full pipeline that classifies the sentiment of financial tweets
and constructs trading signals for AAPL, TSLA and MSFT. Three models are compared:
TF-IDF + LinearSVC (baseline), BiLSTM + GloVe, and fine-tuned FinBERT.

**Sections**
1. Setup & data download
2. Preprocessing
3. Baseline: TF-IDF classifiers + SMOTE experiment
4. BiLSTM with GloVe embeddings
5. FinBERT fine-tuning
6. Synthetic backtest (AAPL)
7. Real-timestamp backtest (AAPL / TSLA / MSFT)

## 1. Setup

In [ ]:
%%capture
!pip install datasets yfinance kaggle transformers torch torchtext imbalanced-learn accelerate --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, json, re, time, shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import FunctionTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from google.colab import drive, userdata
drive.mount('/content/drive')

# Paths
DATA_DIR    = Path('/content/drive/MyDrive/sentiment_project/data')
RAW_DIR     = DATA_DIR / 'raw'
PROC_DIR    = DATA_DIR / 'processed'
METRICS_DIR = DATA_DIR / 'metrics'
MODELS_DIR  = DATA_DIR / 'models'
REAL_DIR    = DATA_DIR / 'real'
for d in [RAW_DIR, PROC_DIR, METRICS_DIR, MODELS_DIR, REAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Labels
LABELS    = ['bearish', 'neutral', 'bullish']
LABEL2IDX = {l: i for i, l in enumerate(LABELS)}
IDX2LABEL = {i: l for l, i in LABEL2IDX.items()}
SCORE_MAP = {'bullish': 1, 'neutral': 0, 'bearish': -1}

sns.set_style('whitegrid')
torch.manual_seed(42)
np.random.seed(42)
print('Setup done.')

## 2. Data

Two corpora are used:
- **HuggingFace** (`zeroshot/twitter-financial-news-sentiment`): 11,931 labelled tweets for training/evaluation
- **Kaggle** (`equinxx/stock-tweets-for-sentiment-analysis-and-prediction`): 80,793 tweets with real UTC timestamps for backtesting

Note: the HuggingFace dataset has no original timestamps. Synthetic timestamps are assigned
to the pre-2021 window so that the training period strictly precedes the Kaggle backtest
period (October 2021 – September 2022).

In [ ]:
# ── HuggingFace corpus ──────────────────────────────────────────────────────
TWEETS_PATH = RAW_DIR / 'tweets_raw.parquet'

if TWEETS_PATH.exists():
    tweets_df = pd.read_parquet(TWEETS_PATH)
    print(f'Loaded from cache: {len(tweets_df):,} tweets')
else:
    ds = load_dataset('zeroshot/twitter-financial-news-sentiment')
    df = pd.concat([ds['train'].to_pandas(), ds['validation'].to_pandas()], ignore_index=True)

    label_map = {0: 'bearish', 1: 'bullish', 2: 'neutral'}
    df['sentiment'] = df['label'].map(label_map)
    df['score']     = df['sentiment'].map(SCORE_MAP)

    # Synthetic timestamps — strictly before Kaggle backtest period (pre Oct 2021)
    trading_days   = pd.bdate_range('2019-01-01', '2021-09-30')
    assigned_days  = np.random.choice(trading_days, size=len(df), replace=True)
    random_minutes = np.random.randint(0, 390, size=len(df))
    df['timestamp'] = [
        pd.Timestamp(day) + pd.Timedelta(minutes=int(570 + m))
        for day, m in zip(assigned_days, random_minutes)
    ]
    df['date'] = df['timestamp'].dt.date

    tweets_df = df
    tweets_df.to_parquet(TWEETS_PATH, index=False)
    print(f'Downloaded and saved: {len(tweets_df):,} tweets')

print(tweets_df['sentiment'].value_counts().to_string())

In [ ]:
# ── AAPL prices (for synthetic backtest) ────────────────────────────────────
AAPL_PATH = RAW_DIR / 'aapl_prices.parquet'

if not AAPL_PATH.exists():
    import yfinance as yf
    aapl = yf.download('AAPL', start='2019-01-01', end='2024-01-31',
                        auto_adjust=True, progress=False)
    if isinstance(aapl.columns, pd.MultiIndex):
        aapl.columns = [c[0].lower() for c in aapl.columns]
    else:
        aapl.columns = [c.lower() for c in aapl.columns]
    aapl.index.name = 'date'
    aapl = aapl[['open', 'high', 'low', 'close', 'volume']].copy()
    aapl['return_fwd1d'] = aapl['close'].pct_change(1).shift(-1)
    aapl.to_parquet(AAPL_PATH)
    print(f'AAPL prices saved: {len(aapl)} rows')
else:
    aapl = pd.read_parquet(AAPL_PATH)
    print(f'AAPL prices loaded: {len(aapl)} rows')

In [ ]:
# ── Kaggle corpus ───────────────────────────────────────────────────────────
KAGGLE_PATH = REAL_DIR / 'tweets_kaggle_raw.parquet'

if not KAGGLE_PATH.exists():
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump({'username': 'tommasozeri', 'key': userdata.get('KAGGLE')}, f)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

    os.system('kaggle datasets download -d equinxx/stock-tweets-for-sentiment-analysis-and-prediction '
              '-p /content/kaggle_data --unzip -q')

    import glob
    dfs = []
    for csv in glob.glob('/content/kaggle_data/**/*.csv', recursive=True):
        df = pd.read_csv(csv, low_memory=False)
        df['_source_file'] = Path(csv).name
        dfs.append(df)
    raw = pd.concat(dfs, ignore_index=True)
    raw.to_parquet(KAGGLE_PATH, index=False)
    print(f'Kaggle data saved: {len(raw):,} rows')
else:
    raw = pd.read_parquet(KAGGLE_PATH)
    print(f'Kaggle data loaded: {len(raw):,} rows')

# Separate tweets and prices
tweets_real = raw[raw['_source_file'] == 'stock_tweets.csv'][
    ['Date', 'Tweet', 'Stock Name']
].rename(columns={'Date': 'date', 'Tweet': 'text', 'Stock Name': 'ticker'}).copy()

prices_real = raw[raw['_source_file'] == 'stock_yfinance_data.csv'][
    ['Date', 'Close', 'Stock Name']
].rename(columns={'Date': 'date', 'Close': 'close', 'Stock Name': 'ticker'}).copy()

print(f'\nTweets by ticker:')
print(tweets_real['ticker'].value_counts().head(8).to_string())

## 3. Preprocessing

Two strategies: aggressive normalisation for TF-IDF/BiLSTM, minimal for FinBERT.
The key difference is that FinBERT is case-sensitive and uses punctuation as syntactic cues.

In [ ]:
def preprocess_tfidf(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\$[A-Za-z]+', 'ticker', text)   # $AAPL -> ticker (generalise)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def preprocess_bert(text: str) -> str:
    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\$([A-Za-z]+)', r'\1', text)   # $AAPL -> AAPL (keep case)
    return re.sub(r'\s+', ' ', text).strip()

# ── Train / val / test split — fixed seed, used by all models ──────────────
X_raw = tweets_df['text'].values
y_raw = tweets_df['sentiment'].values

X_train_raw, X_temp, y_train, y_temp = train_test_split(
    X_raw, y_raw, test_size=0.30, random_state=42, stratify=y_raw
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# TF-IDF version
X_train = np.array([preprocess_tfidf(t) for t in X_train_raw])
X_val   = np.array([preprocess_tfidf(t) for t in X_val_raw])
X_test  = np.array([preprocess_tfidf(t) for t in X_test_raw])

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Class distribution (train): {dict(zip(*np.unique(y_train, return_counts=True)))}')

## 4. Baseline: TF-IDF + Classifiers

Three pipelines: Logistic Regression, LinearSVC, and SVM with RBF kernel.
TF-IDF uses bigrams (`ngram_range=(1,2)`) to capture phrases like "strong buy" or "below expectations".

In [ ]:
TFIDF_PARAMS = dict(ngram_range=(1, 2), max_features=50_000,
                    sublinear_tf=True, min_df=2)

models = {
    'TF-IDF + LR': Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   LogisticRegression(max_iter=1000, C=1.0,
                                      class_weight='balanced', random_state=42))
    ]),
    'TF-IDF + LinearSVC': Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   LinearSVC(C=1.0, class_weight='balanced',
                             max_iter=2000, random_state=42))
    ]),
    'TF-IDF + SVM (RBF)': Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   SVC(kernel='rbf', C=1.0, class_weight='balanced',
                       random_state=42, probability=True))
    ]),
}

def evaluate(name, pipeline, Xtr, ytr, Xte, yte):
    t0 = time.time()
    pipeline.fit(Xtr, ytr)
    train_t = time.time() - t0
    t0 = time.time()
    y_pred = pipeline.predict(Xte)
    infer_t = (time.time() - t0) / len(Xte) * 1000
    acc  = accuracy_score(yte, y_pred)
    f1m  = f1_score(yte, y_pred, average='macro')
    f1c  = f1_score(yte, y_pred, average=None, labels=LABELS)
    print(f'\n{name}')
    print(f'  Accuracy: {acc:.4f}  F1 macro: {f1m:.4f}  Train: {train_t:.1f}s')
    print(classification_report(yte, y_pred, target_names=LABELS))
    return {'model': name, 'accuracy': round(acc, 4), 'f1_macro': round(f1m, 4),
            'f1_bearish': round(f1c[0], 4), 'f1_neutral': round(f1c[1], 4),
            'f1_bullish': round(f1c[2], 4), 'train_time_s': round(train_t, 2),
            'infer_ms': round(infer_t, 3), 'y_pred': y_pred.tolist()}

results = {}
for name, pipe in models.items():
    results[name] = evaluate(name, pipe, X_train, y_train, X_test, y_test)

In [ ]:
# ── Confusion matrices ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'], labels=LABELS)
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)
    ax.tick_params(axis='x', labelrotation=20)
plt.suptitle('Baseline models — Confusion Matrix', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(METRICS_DIR / '01_baseline_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── SMOTE experiment ────────────────────────────────────────────────────────
# SMOTE must be applied after TF-IDF (on numeric vectors), not on raw text.
# Result: negligible improvement — the imbalance problem is semantic, not statistical.

to_dense = FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)

models_smote = {
    'TF-IDF + LR + SMOTE': ImbPipeline([
        ('tfidf',    TfidfVectorizer(**TFIDF_PARAMS)),
        ('to_dense', to_dense),
        ('smote',    SMOTE(random_state=42, k_neighbors=5)),
        ('clf',      LogisticRegression(max_iter=1000, C=1.0, random_state=42))
    ]),
    'TF-IDF + LinearSVC + SMOTE': ImbPipeline([
        ('tfidf',    TfidfVectorizer(**TFIDF_PARAMS)),
        ('to_dense', to_dense),
        ('smote',    SMOTE(random_state=42, k_neighbors=5)),
        ('clf',      LinearSVC(C=1.0, max_iter=2000, random_state=42))
    ]),
}

results_smote = {}
for name, pipe in models_smote.items():
    results_smote[name] = evaluate(name, pipe, X_train, y_train, X_test, y_test)

# Save baseline metrics
metrics_to_save = [{k: v for k, v in r.items() if k != 'y_pred'}
                   for r in {**results, **results_smote}.values()]
with open(METRICS_DIR / '01_baseline.json', 'w') as f:
    json.dump(metrics_to_save, f, indent=2)
print('Saved: 01_baseline.json')

## 5. BiLSTM with GloVe Embeddings

Architecture: GloVe Twitter 27B (100d) → Dropout → BiLSTM(128, 2 layers) → Dropout → Linear(256→3)

Bidirectionality captures both left-to-right and right-to-left context simultaneously,
which matters for financial sentences like "revenue below expectations despite strong volume".

In [ ]:
# ── Vocabulary from training set only (avoid leakage) ───────────────────────
MAX_VOCAB = 20_000
MAX_LEN   = 64
PAD_IDX, UNK_IDX = 0, 1

counter = Counter()
for text in X_train:
    counter.update(text.split())

vocab = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
for word, _ in counter.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)

def encode(text):
    tokens = text.split()[:MAX_LEN]
    ids    = [vocab.get(t, UNK_IDX) for t in tokens]
    ids   += [PAD_IDX] * (MAX_LEN - len(ids))
    return ids

print(f'Vocabulary size: {len(vocab):,}')

In [ ]:
# ── GloVe embeddings ────────────────────────────────────────────────────────
EMBED_DIM   = 100
GLOVE_PATH  = Path('/content/glove.twitter.27B.100d.txt')
GLOVE_DRIVE = DATA_DIR / 'glove.twitter.27B.100d.txt'

if GLOVE_DRIVE.exists():
    shutil.copy(GLOVE_DRIVE, GLOVE_PATH)
    print('GloVe loaded from Drive cache')
else:
    print('Downloading GloVe Twitter 27B...')
    os.system('wget -q --show-progress https://nlp.stanford.edu/data/glove.twitter.27B.zip')
    os.system('unzip -q glove.twitter.27B.zip glove.twitter.27B.100d.txt')
    shutil.copy(GLOVE_PATH, GLOVE_DRIVE)
    print('GloVe saved to Drive for future sessions')

glove_vecs = {}
with open(GLOVE_PATH, encoding='utf-8') as f:
    for line in f:
        parts = line.rstrip().split()
        if parts[0] in vocab:
            glove_vecs[parts[0]] = np.array(parts[1:], dtype=np.float32)

embedding_matrix = np.zeros((len(vocab), EMBED_DIM), dtype=np.float32)
for word, idx in vocab.items():
    if word in glove_vecs:
        embedding_matrix[idx] = glove_vecs[word]
    elif idx > 1:
        embedding_matrix[idx] = np.random.normal(0, 0.1, EMBED_DIM)

print(f'GloVe coverage: {len(glove_vecs):,} / {len(vocab):,} ({len(glove_vecs)/len(vocab)*100:.1f}%)')

In [ ]:
# ── Dataset and DataLoader ───────────────────────────────────────────────────
class TweetDataset(Dataset):
    def __init__(self, texts, labels):
        self.X = [torch.tensor(encode(t), dtype=torch.long) for t in texts]
        self.y = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

y_train_idx = np.array([LABEL2IDX[l] for l in y_train])
y_val_idx   = np.array([LABEL2IDX[l] for l in y_val])
y_test_idx  = np.array([LABEL2IDX[l] for l in y_test])

BATCH = 64
train_loader = DataLoader(TweetDataset(X_train, y_train_idx), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TweetDataset(X_val,   y_val_idx),   batch_size=BATCH)
test_loader  = DataLoader(TweetDataset(X_test,  y_test_idx),  batch_size=BATCH)

In [ ]:
# ── BiLSTM model ────────────────────────────────────────────────────────────
class BiLSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers,
                 num_classes, dropout, emb_matrix=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        if emb_matrix is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(emb_matrix))
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                             batch_first=True, bidirectional=True,
                             dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(emb)
        out = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.fc(self.dropout(out))

bilstm = BiLSTMSentiment(
    vocab_size=len(vocab), embed_dim=EMBED_DIM,
    hidden_dim=128, num_layers=2, num_classes=3,
    dropout=0.3, emb_matrix=embedding_matrix
).to(DEVICE)

print(f'Parameters: {sum(p.numel() for p in bilstm.parameters()):,}')

In [ ]:
# ── Training ─────────────────────────────────────────────────────────────────
cw = compute_class_weight('balanced', classes=np.array([0,1,2]), y=y_train_idx)
criterion_bilstm = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float).to(DEVICE))
optimizer_bilstm = Adam(bilstm.parameters(), lr=1e-3, weight_decay=1e-5)

def train_epoch_bilstm(model, loader, criterion, optimizer):
    model.train()
    total_loss = total = correct = 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(y_b)
        correct    += (model(X_b).argmax(1) == y_b).sum().item()
        total      += len(y_b)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch_bilstm(model, loader, criterion):
    model.eval()
    total_loss = total = correct = 0
    preds, labels = [], []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        out = model(X_b)
        total_loss += criterion(out, y_b).item() * len(y_b)
        p = out.argmax(1)
        correct += (p == y_b).sum().item()
        total   += len(y_b)
        preds.extend(p.cpu().numpy()); labels.extend(y_b.cpu().numpy())
    return total_loss/total, correct/total, f1_score(labels, preds, average='macro'), preds

best_bilstm_f1   = 0
best_bilstm_path = MODELS_DIR / 'bilstm_best.pt'
history_bilstm   = {'train_loss': [], 'val_loss': [], 'val_f1': []}
t_bilstm = time.time()

for epoch in range(1, 16):
    tl, ta = train_epoch_bilstm(bilstm, train_loader, criterion_bilstm, optimizer_bilstm)
    vl, va, vf1, _ = eval_epoch_bilstm(bilstm, val_loader, criterion_bilstm)
    history_bilstm['train_loss'].append(tl)
    history_bilstm['val_loss'].append(vl)
    history_bilstm['val_f1'].append(vf1)
    if vf1 > best_bilstm_f1:
        best_bilstm_f1 = vf1
        torch.save(bilstm.state_dict(), best_bilstm_path)
        flag = ' ← best'
    else:
        flag = ''
    print(f'Epoch {epoch:02d}/15 | train {tl:.4f} | val {vl:.4f} f1 {vf1:.4f}{flag}')

bilstm_train_time = time.time() - t_bilstm
print(f'\nDone in {bilstm_train_time:.1f}s  |  Best val F1: {best_bilstm_f1:.4f}')

In [ ]:
# ── Test evaluation ──────────────────────────────────────────────────────────
bilstm.load_state_dict(torch.load(best_bilstm_path, map_location=DEVICE))
t0 = time.time()
_, bilstm_acc, bilstm_f1, bilstm_preds = eval_epoch_bilstm(bilstm, test_loader, criterion_bilstm)
bilstm_infer = (time.time() - t0) / len(X_test) * 1000

y_bilstm_labels = [IDX2LABEL[i] for i in bilstm_preds]
f1_bilstm_each  = f1_score(y_test, y_bilstm_labels, average=None, labels=LABELS)

print(f'Accuracy:  {bilstm_acc:.4f}')
print(f'F1 Macro:  {bilstm_f1:.4f}')
print(classification_report(y_test, y_bilstm_labels, target_names=LABELS))

# Save metrics
bilstm_metrics = {
    'model': 'BiLSTM + GloVe',
    'accuracy': round(bilstm_acc, 4), 'f1_macro': round(bilstm_f1, 4),
    'f1_bearish': round(f1_bilstm_each[0], 4),
    'f1_neutral':  round(f1_bilstm_each[1], 4),
    'f1_bullish':  round(f1_bilstm_each[2], 4),
    'train_time_s': round(bilstm_train_time, 2),
    'infer_ms': round(bilstm_infer, 3),
}
with open(METRICS_DIR / '02_bilstm.json', 'w') as f:
    json.dump(bilstm_metrics, f, indent=2)
print('Saved: 02_bilstm.json')

## 6. FinBERT Fine-tuning

`ProsusAI/finbert` is BERT pre-trained on 1.8B words of financial text (Reuters, Bloomberg, SEC filings).
Fine-tuning uses a two-phase strategy to prevent catastrophic forgetting:
1. **Warmup (1 epoch):** only the classification head is trained
2. **Full fine-tuning (4 epochs):** all layers with lr=2e-5 and linear warmup

In [ ]:
MODEL_NAME = 'ProsusAI/finbert'
MAX_LEN_BERT = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# BERT preprocessing — minimal, preserve case and punctuation
X_train_bert = np.array([preprocess_bert(t) for t in X_train_raw])
X_val_bert   = np.array([preprocess_bert(t) for t in X_val_raw])
X_test_bert  = np.array([preprocess_bert(t) for t in X_test_raw])

class FinBERTDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), max_length=MAX_LEN_BERT,
                              padding='max_length', truncation=True, return_tensors='pt')
        self.labels = torch.tensor([LABEL2IDX[l] for l in labels], dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {'input_ids': self.enc['input_ids'][i],
                'attention_mask': self.enc['attention_mask'][i],
                'labels': self.labels[i]}

BATCH_BERT = 32
print('Tokenising...')
train_bert = DataLoader(FinBERTDataset(X_train_bert, y_train), batch_size=BATCH_BERT,
                         shuffle=True, pin_memory=True, num_workers=2)
val_bert   = DataLoader(FinBERTDataset(X_val_bert,   y_val),   batch_size=BATCH_BERT,
                         pin_memory=True, num_workers=2)
test_bert  = DataLoader(FinBERTDataset(X_test_bert,  y_test),  batch_size=BATCH_BERT,
                         pin_memory=True, num_workers=2)
print('Done.')

In [ ]:
finbert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=IDX2LABEL, label2id=LABEL2IDX,
    ignore_mismatched_sizes=True
).to(DEVICE)

cw_bert = compute_class_weight('balanced', classes=np.array([0,1,2]),
                                 y=np.array([LABEL2IDX[l] for l in y_train]))
criterion_bert = nn.CrossEntropyLoss(
    weight=torch.tensor(cw_bert, dtype=torch.float).to(DEVICE)
)

def train_epoch_bert(model, loader, optimizer, scheduler):
    model.train()
    total_loss = total = correct = 0
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbl  = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        out  = model(input_ids=ids, attention_mask=mask)
        loss = criterion_bert(out.logits, lbl)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item() * len(lbl)
        correct    += (out.logits.argmax(1) == lbl).sum().item()
        total      += len(lbl)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch_bert(model, loader):
    model.eval()
    total_loss = total = correct = 0
    preds, labels = [], []
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbl  = batch['labels'].to(DEVICE)
        out  = model(input_ids=ids, attention_mask=mask)
        total_loss += criterion_bert(out.logits, lbl).item() * len(lbl)
        p = out.logits.argmax(1)
        correct += (p == lbl).sum().item(); total += len(lbl)
        preds.extend(p.cpu().numpy()); labels.extend(lbl.cpu().numpy())
    return total_loss/total, correct/total, f1_score(labels, preds, average='macro'), preds

In [ ]:
# ── Phase 1: warmup — classifier head only ───────────────────────────────────
print('Phase 1: warmup (1 epoch, head only)')
for p in finbert.bert.parameters():
    p.requires_grad = False

opt_warmup  = AdamW(filter(lambda p: p.requires_grad, finbert.parameters()), lr=1e-3)
sch_warmup  = get_linear_schedule_with_warmup(opt_warmup, 0, len(train_bert))

tl, ta = train_epoch_bert(finbert, train_bert, opt_warmup, sch_warmup)
vl, va, vf1, _ = eval_epoch_bert(finbert, val_bert)
print(f'Warmup — train {tl:.4f} | val {vl:.4f} f1 {vf1:.4f}')

In [ ]:
# ── Phase 2: full fine-tuning ─────────────────────────────────────────────────
print('Phase 2: full fine-tuning (4 epochs, lr=2e-5)')
for p in finbert.bert.parameters():
    p.requires_grad = True

EPOCHS_BERT  = 4
total_steps  = len(train_bert) * EPOCHS_BERT
warmup_steps = int(total_steps * 0.1)

opt_ft = AdamW(finbert.parameters(), lr=2e-5, weight_decay=0.01)
sch_ft = get_linear_schedule_with_warmup(opt_ft, warmup_steps, total_steps)

best_bert_f1   = 0
best_bert_path = MODELS_DIR / 'finbert_best.pt'
history_bert   = {'train_loss': [], 'val_loss': [], 'val_f1': []}
t_bert         = time.time()
patience_cnt   = 0

for epoch in range(1, EPOCHS_BERT + 1):
    tl, ta = train_epoch_bert(finbert, train_bert, opt_ft, sch_ft)
    vl, va, vf1, _ = eval_epoch_bert(finbert, val_bert)
    history_bert['train_loss'].append(tl)
    history_bert['val_loss'].append(vl)
    history_bert['val_f1'].append(vf1)
    if vf1 > best_bert_f1:
        best_bert_f1 = vf1; patience_cnt = 0
        torch.save(finbert.state_dict(), best_bert_path); flag = ' ← best'
    else:
        patience_cnt += 1; flag = f' (patience {patience_cnt}/2)'
    print(f'Epoch {epoch}/{EPOCHS_BERT} | train {tl:.4f} acc {ta:.4f} | val {vl:.4f} f1 {vf1:.4f}{flag}')
    if patience_cnt >= 2:
        print('Early stopping.')
        break

bert_train_time = time.time() - t_bert
print(f'\nDone in {bert_train_time:.1f}s  |  Best val F1: {best_bert_f1:.4f}')

In [ ]:
# ── Test evaluation ──────────────────────────────────────────────────────────
finbert.load_state_dict(torch.load(best_bert_path, map_location=DEVICE))
t0 = time.time()
_, bert_acc, bert_f1, bert_preds = eval_epoch_bert(finbert, test_bert)
bert_infer = (time.time() - t0) / len(X_test) * 1000

y_bert_labels = [IDX2LABEL[i] for i in bert_preds]
f1_bert_each  = f1_score(y_test, y_bert_labels, average=None, labels=LABELS)

print(f'Accuracy: {bert_acc:.4f}  F1 Macro: {bert_f1:.4f}')
print(classification_report(y_test, y_bert_labels, target_names=LABELS))

# ── Full model comparison ─────────────────────────────────────────────────────
with open(METRICS_DIR / '01_baseline.json') as f:
    m01 = json.load(f)
best_svc = next(m for m in m01 if m['model'] == 'TF-IDF + LinearSVC')

finbert_metrics = {
    'model': 'FinBERT', 'accuracy': round(bert_acc, 4), 'f1_macro': round(bert_f1, 4),
    'f1_bearish': round(f1_bert_each[0], 4), 'f1_neutral': round(f1_bert_each[1], 4),
    'f1_bullish': round(f1_bert_each[2], 4),
    'train_time_s': round(bert_train_time, 2), 'infer_ms': round(bert_infer, 3),
}
with open(METRICS_DIR / '03_finbert.json', 'w') as f:
    json.dump(finbert_metrics, f, indent=2)

compare = pd.DataFrame([best_svc, bilstm_metrics, finbert_metrics])[
    ['model','f1_macro','f1_bearish','f1_neutral','f1_bullish','train_time_s']]
print('\n' + compare.to_string(index=False))

In [ ]:
# ── Training curves ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(history_bert['train_loss'], label='train')
axes[0].plot(history_bert['val_loss'],   label='val')
axes[0].set_title('FinBERT Loss'); axes[0].legend()

axes[1].plot(history_bert['val_f1'], marker='o', color='green')
axes[1].set_title('FinBERT Val F1 Macro')

cm = confusion_matrix(y_test, y_bert_labels, labels=LABELS)
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=axes[2], colorbar=False, cmap='Blues')
axes[2].set_title('FinBERT — Test Confusion Matrix')
axes[2].tick_params(axis='x', labelrotation=20)

for ax in axes[:2]:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.suptitle('FinBERT fine-tuned', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(METRICS_DIR / '03_finbert_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Synthetic Backtest (AAPL)

The HuggingFace tweets have synthetic timestamps, so this backtest is not valid for
signal evaluation — it demonstrates the end-to-end pipeline and motivates the need
for real timestamps.

In [ ]:
# ── Inference on all HuggingFace tweets ─────────────────────────────────────
finbert.eval()
tweets_df['text_bert'] = tweets_df['text'].apply(preprocess_bert)

class InferDataset(Dataset):
    def __init__(self, texts):
        self.enc = tokenizer(list(texts), max_length=MAX_LEN_BERT,
                              padding='max_length', truncation=True, return_tensors='pt')
    def __len__(self): return self.enc['input_ids'].shape[0]
    def __getitem__(self, i):
        return {'input_ids': self.enc['input_ids'][i],
                'attention_mask': self.enc['attention_mask'][i]}

infer_loader = DataLoader(InferDataset(tweets_df['text_bert'].values),
                           batch_size=128, num_workers=2)
all_probs, all_preds = [], []
with torch.no_grad():
    for batch in infer_loader:
        logits = finbert(input_ids=batch['input_ids'].to(DEVICE),
                          attention_mask=batch['attention_mask'].to(DEVICE)).logits
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(probs.argmax(axis=1))

probs_arr = np.array(all_probs)
tweets_df['pred_label']     = [IDX2LABEL[i] for i in all_preds]
tweets_df['weighted_score'] = probs_arr[:, 2] - probs_arr[:, 0]
tweets_df['date_dt']        = pd.to_datetime(tweets_df['date'])
print(f'Inference done: {len(tweets_df):,} tweets')

In [ ]:
# ── Daily sentiment index ────────────────────────────────────────────────────
daily = (tweets_df.groupby('date_dt')
         .agg(sentiment_index=('pred_label',
                lambda x: (x=='bullish').mean() - (x=='bearish').mean()),
              weighted_score=('weighted_score', 'mean'))
         .reset_index().rename(columns={'date_dt': 'date'})
         .sort_values('date'))

daily['sentiment_ma3'] = daily['sentiment_index'].rolling(3, min_periods=1).mean()

# Merge with prices
prices_bt = aapl.reset_index().copy()
prices_bt['date'] = pd.to_datetime(prices_bt['date']).dt.tz_localize(None)
daily['date']     = pd.to_datetime(daily['date']).dt.tz_localize(None)

df_bt = pd.merge(prices_bt, daily, on='date', how='inner').sort_values('date')
df_bt['signal_lag1'] = df_bt['sentiment_index'].shift(1)
df_bt['weighted_lag1'] = df_bt['weighted_score'].shift(1)
df_bt = df_bt.dropna(subset=['return_fwd1d', 'signal_lag1'])
print(f'Aligned rows: {len(df_bt):,}')

In [ ]:
# ── Backtest function ────────────────────────────────────────────────────────
def backtest(df, signal_col, threshold=0.05, tc=0.001):
    sig = df[signal_col].values
    ret = df['return_fwd1d'].values
    pos = np.where(sig > threshold, 1, np.where(sig < -threshold, -1, 0))
    strat = pos * ret - (np.diff(pos, prepend=0) != 0).astype(float) * tc
    cum_s   = pd.Series((1 + strat).cumprod(), index=df['date'].values)
    cum_bah = pd.Series((1 + ret).cumprod(),   index=df['date'].values)
    sharpe  = strat.mean() / strat.std() * np.sqrt(252) if strat.std() > 0 else 0
    max_dd  = ((cum_s / cum_s.cummax()) - 1).min()
    hit     = (strat[pos != 0] > 0).mean() if (pos != 0).any() else 0
    return {'total_return_%': round((cum_s.iloc[-1]-1)*100, 2),
            'bah_return_%':   round((cum_bah.iloc[-1]-1)*100, 2),
            'sharpe': round(sharpe, 3), 'max_dd_%': round(max_dd*100, 2),
            'hit_rate_%': round(hit*100, 2),
            'cum_strategy': cum_s, 'cum_bah': cum_bah}

res_synth = backtest(df_bt, 'signal_lag1', threshold=0.05)
print(f"Synthetic backtest — FinBERT signal: {res_synth['total_return_%']:+.1f}%  "
      f"B&H: {res_synth['bah_return_%']:+.1f}%  Sharpe: {res_synth['sharpe']:.3f}")
print("(Timestamps are synthetic — this result is not valid for signal evaluation)")

## 8. Real-Timestamp Backtest (AAPL / TSLA / MSFT)

Both LinearSVC and FinBERT are applied to the Kaggle corpus — tweets never seen during training.
Three signals are compared: LinearSVC, FinBERT (binary), and Weighted FinBERT (confidence-weighted).

**Research question:** does a more accurate classifier produce a better trading signal?

In [ ]:
TICKERS = ['AAPL', 'TSLA', 'MSFT']

# ── Filter and parse timestamps ───────────────────────────────────────────────
tweets_k = tweets_real[tweets_real['ticker'].isin(TICKERS)].copy()
prices_k  = prices_real[prices_real['ticker'].isin(TICKERS)].copy()

tweets_k['date'] = pd.to_datetime(tweets_k['date'], utc=True).dt.tz_localize(None)
tweets_k['trade_date'] = tweets_k['date'].dt.normalize()
tweets_k = tweets_k.dropna(subset=['date', 'text'])

prices_k['date'] = pd.to_datetime(prices_k['date'], utc=True).dt.tz_localize(None).dt.normalize()
prices_k = prices_k.dropna(subset=['close'])

print(f'Kaggle tweets filtered: {len(tweets_k):,}')
print(tweets_k['ticker'].value_counts().to_string())

In [ ]:
# ── FinBERT inference on Kaggle tweets ───────────────────────────────────────
tweets_k['text_bert'] = tweets_k['text'].apply(preprocess_bert)

infer_k = DataLoader(InferDataset(tweets_k['text_bert'].values),
                      batch_size=128, num_workers=2)
probs_k, preds_k = [], []
with torch.no_grad():
    for batch in infer_k:
        logits = finbert(input_ids=batch['input_ids'].to(DEVICE),
                          attention_mask=batch['attention_mask'].to(DEVICE)).logits
        p = torch.softmax(logits, dim=1).cpu().numpy()
        probs_k.extend(p); preds_k.extend(p.argmax(axis=1))

probs_k_arr = np.array(probs_k)
tweets_k = tweets_k.copy()
tweets_k['pred_label']     = [IDX2LABEL[i] for i in preds_k]
tweets_k['weighted_score'] = probs_k_arr[:, 2] - probs_k_arr[:, 0]
print(f'FinBERT inference done: {len(tweets_k):,} tweets')

In [ ]:
# ── LinearSVC inference on Kaggle tweets ─────────────────────────────────────
# Retrain on HuggingFace training split (same hyperparams, same seed)
from datasets import load_dataset as lds
ds2 = lds('zeroshot/twitter-financial-news-sentiment')
hf  = pd.concat([ds2['train'].to_pandas(), ds2['validation'].to_pandas()], ignore_index=True)
hf['sentiment'] = hf['label'].map({0:'bearish',1:'bullish',2:'neutral'})
hf['text_clean'] = hf['text'].apply(preprocess_tfidf)

X_hf_tr, _, y_hf_tr, _ = train_test_split(
    hf['text_clean'].values, hf['sentiment'].values,
    test_size=0.30, random_state=42, stratify=hf['sentiment'].values
)
svc_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
    ('clf',   LinearSVC(C=1.0, class_weight='balanced', max_iter=2000, random_state=42))
])
svc_pipe.fit(X_hf_tr, y_hf_tr)

tweets_k['text_tfidf'] = tweets_k['text'].apply(preprocess_tfidf)
tweets_k['svc_label']  = svc_pipe.predict(tweets_k['text_tfidf'].values)
tweets_k['svc_score']  = tweets_k['svc_label'].map(SCORE_MAP)
print('LinearSVC inference done.')

In [ ]:
# ── Daily sentiment indices ───────────────────────────────────────────────────
def daily_index(df, label_col, score_col=None):
    grp = df.groupby(['trade_date', 'ticker'])
    si  = grp[label_col].apply(
        lambda x: (x=='bullish').mean() - (x=='bearish').mean()).reset_index()
    si.columns = ['date', 'ticker', 'sentiment_index']
    if score_col:
        ws = grp[score_col].mean().reset_index()
        ws.columns = ['date', 'ticker', 'weighted_score']
        si = si.merge(ws, on=['date','ticker'])
    si['date'] = pd.to_datetime(si['date'])
    return si.sort_values(['ticker','date'])

daily_finbert_k = daily_index(tweets_k, 'pred_label', 'weighted_score')
daily_svc_k     = daily_index(tweets_k, 'svc_label')

# ── Merge with prices ─────────────────────────────────────────────────────────
prices_k = prices_k.sort_values(['ticker','date'])
prices_k['return_fwd1d'] = prices_k.groupby('ticker')['close'].pct_change(1).shift(-1)

df_fb  = prices_k.merge(daily_finbert_k, on=['date','ticker'], how='inner')
df_svc = prices_k.merge(daily_svc_k[['date','ticker','sentiment_index']]
                          .rename(columns={'sentiment_index':'svc_index'}),
                          on=['date','ticker'], how='inner')

df_real = df_fb.merge(df_svc[['date','ticker','svc_index']], on=['date','ticker'], how='inner')
df_real['finbert_lag1']  = df_real.groupby('ticker')['sentiment_index'].shift(1)
df_real['weighted_lag1'] = df_real.groupby('ticker')['weighted_score'].shift(1)
df_real['svc_lag1']      = df_real.groupby('ticker')['svc_index'].shift(1)
df_real = df_real.dropna(subset=['return_fwd1d','finbert_lag1','svc_lag1'])

print(f'Aligned rows: {len(df_real):,}')
print(df_real.groupby('ticker')[['date']].agg(['min','max','count']).to_string())

In [ ]:
# ── Backtest — all models, all tickers ────────────────────────────────────────
signals = {
    'LinearSVC':       'svc_lag1',
    'FinBERT':         'finbert_lag1',
    'Weighted FinBERT':'weighted_lag1',
}

all_results = {}
print(f"{'Ticker':<8} {'Model':<22} {'Return%':>10} {'B&H%':>7} {'Sharpe':>8} {'Hit%':>7}")
print('-' * 65)

for ticker in TICKERS:
    df_t = df_real[df_real['ticker'] == ticker]
    if len(df_t) < 20:
        continue
    all_results[ticker] = {}
    for mname, scol in signals.items():
        res = backtest(df_t, scol, threshold=0.05)
        all_results[ticker][mname] = res
        print(f"{ticker:<8} {mname:<22} {res['total_return_%']:>+10.2f} "
              f"{res['bah_return_%']:>+7.2f} {res['sharpe']:>8.3f} "
              f"{res['hit_rate_%']:>7.2f}")
    print()

In [ ]:
# ── Pearson correlations ─────────────────────────────────────────────────────
print(f"{'Ticker':<8} {'LinearSVC':>12} {'FinBERT':>10} {'Weighted':>10}")
print('-' * 44)
for ticker in TICKERS:
    df_t = df_real[df_real['ticker'] == ticker]
    if len(df_t) < 10: continue
    r1 = df_t['svc_lag1'].corr(df_t['return_fwd1d'])
    r2 = df_t['finbert_lag1'].corr(df_t['return_fwd1d'])
    r3 = df_t['weighted_lag1'].corr(df_t['return_fwd1d'])
    print(f"{ticker:<8} {r1:>+12.4f} {r2:>+10.4f} {r3:>+10.4f}")

print("\nNote: correlations > |0.05| are economically significant on daily data.")

In [ ]:
# ── Cumulative return plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(len(all_results), 1, figsize=(13, 4.5*len(all_results)))
if len(all_results) == 1: axes = [axes]

colors = {'LinearSVC': '#B5D4F4', 'FinBERT': '#185FA5', 'Weighted FinBERT': '#1D9E75'}

for ax, (ticker, res_dict) in zip(axes, all_results.items()):
    bah = list(res_dict.values())[0]['cum_bah']
    ax.plot(bah.index, bah.values, color='#888780', lw=1.5, ls='--', label='Buy & Hold')
    for mname, res in res_dict.items():
        ax.plot(res['cum_strategy'].index, res['cum_strategy'].values,
                color=colors[mname], lw=2, label=f"{mname} ({res['total_return_%']:+.1f}%)")
    bah_ret = list(res_dict.values())[0]['bah_return_%']
    ax.set_title(f'{ticker} — B&H: {bah_ret:+.1f}%', fontsize=11)
    ax.set_ylabel('Portfolio value (base 1.0)')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('Real-Timestamp Backtest — LinearSVC vs FinBERT vs Weighted FinBERT',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(METRICS_DIR / '05_real_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('=' * 65)
print('CLASSIFICATION RESULTS')
print('=' * 65)
with open(METRICS_DIR / '01_baseline.json') as f: m01 = json.load(f)
with open(METRICS_DIR / '02_bilstm.json')  as f: m02 = json.load(f)
with open(METRICS_DIR / '03_finbert.json') as f: m03 = json.load(f)

best_svc = next(m for m in m01 if m['model'] == 'TF-IDF + LinearSVC')
clf_df = pd.DataFrame([best_svc, m02, m03])[
    ['model','f1_macro','f1_bearish','f1_neutral','f1_bullish','train_time_s']]
print(clf_df.to_string(index=False))

print('\n' + '=' * 65)
print('BACKTEST RESULTS (threshold=0.05, lag=+1 day, TC=0.1%)')
print('=' * 65)
rows = []
for ticker, rd in all_results.items():
    bah = list(rd.values())[0]['bah_return_%']
    for mname, res in rd.items():
        rows.append({'ticker': ticker, 'model': mname,
                     'return_%': res['total_return_%'], 'bah_%': bah,
                     'sharpe': res['sharpe'], 'hit_%': res['hit_rate_%']})
    rows.append({'ticker': '', 'model': 'Buy & Hold', 'return_%': bah,
                 'bah_%': bah, 'sharpe': '-', 'hit_%': '-'})
print(pd.DataFrame(rows).to_string(index=False))